# Продажи дистрибьютора: что стоит за ростом на 35%

Данные: 301 355 строк отгрузок за 04.01.2018 — 31.08.2018. Пять складов, 211 контрагентов, 24 позиции номенклатуры, 889 467 единиц продукции.

**Вопрос:** продажи за период выросли на 35%. Нужно понять, за счёт чего именно, и проверить популярное объяснение — что спрос зависит от погоды.

**Что делаем:**
1. Проверяем качество данных до анализа
2. Дневная динамика и её описание
3. Поиск выбросов — на уровне строки и на уровне дня
4. Точка перелома тренда
5. Топовый товар по срезу: склад 3, среды, июнь — август
6. ABC-анализ номенклатуры и концентрация клиентов
7. Проверка гипотезы о влиянии температуры

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .3, "grid.linestyle": "--",
    "figure.dpi": 110,
})
NAVY, ORANGE, GREEN, GREY = "#1F3864", "#C55A11", "#548235", "#808080"
RU_MON = {1: "янв", 2: "фев", 3: "мар", 4: "апр",
          5: "май", 6: "июн", 7: "июл", 8: "авг", 9: "сен"}
DAYS = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]

## 1. Загрузка и первый взгляд

In [ ]:
df = pd.read_csv("data/data.csv")
df["Дата"] = pd.to_datetime(df["Дата"])

print(f"строк: {len(df):,}".replace(",", " "))
print(f"период: {df['Дата'].min():%d.%m.%Y} — {df['Дата'].max():%d.%m.%Y}")
df.head()

In [ ]:
df.info()

## 2. Проверки качества данных

Эти проверки идут до анализа, а не после. Незамеченный пропуск в календаре или дубликат ключа
искажает любую последующую агрегацию, и обнаруживается это обычно уже после того,
как цифры ушли в отчёт.

In [ ]:
checks = {
    "строк": len(df),
    "пропусков всего": int(df.isna().sum().sum()),
    "полных дублей строк": int(df.duplicated().sum()),
    "дублей ключа Дата+Склад+Контрагент+Номенклатура":
        int(df.duplicated(["Дата", "Склад", "Контрагент", "Номенклатура"]).sum()),
    "отрицательных количеств": int((df["Количество"] < 0).sum()),
    "нулевых количеств": int((df["Количество"] == 0).sum()),
    "уникальных дат": df["Дата"].nunique(),
    "складов": df["Склад"].nunique(),
    "контрагентов": df["Контрагент"].nunique(),
    "номенклатур": df["Номенклатура"].nunique(),
}
for k, v in checks.items():
    print(f"{k:52s} {v:>10,}".replace(",", " "))

Дубликатов и пропусков нет. Но два результата требуют внимания:
нулевых количеств 31 425 — это 10.4% строк, и уникальных дат 205,
хотя период охватывает 240 календарных дней.

In [ ]:
# Каких дней не хватает в календаре
full_range = pd.date_range(df["Дата"].min(), df["Дата"].max())
missing = sorted(set(full_range) - set(df["Дата"].unique()))

print(f"пропущено дней: {len(missing)} из {len(full_range)}")
by_dow = pd.Series([d.dayofweek for d in missing]).value_counts().sort_index()
print("\nпо дням недели:")
for dow, cnt in by_dow.items():
    print(f"  {DAYS[dow]}: {cnt}")
print("\nпропуски, не являющиеся понедельником:",
      [f"{d:%d.%m.%Y}" for d in missing if d.dayofweek != 0])

**Понедельников в данных нет вообще** — пропущены все 34 понедельника периода.
Это не потеря данных, а режим работы: отгрузок по понедельникам не происходит.

Следствия, которые нужно держать в голове весь дальнейший анализ:
недельная сезонность считается по шести дням, а не по семи;
любое «среднее за неделю» — это среднее за шесть рабочих дней;
пропуск 22.03.2018 (четверг) — единственный настоящий пробел в данных.

In [ ]:
# Нулевые количества: это ошибка или закономерность?
zero_share = (df["Количество"] == 0).mean()
print(f"доля нулевых строк: {zero_share:.1%}\n")

print("по складам:")
print(df.groupby("Склад")["Количество"].apply(lambda x: (x == 0).mean())
        .mul(100).round(1).to_string())

print("\nпо месяцам:")
by_month = df.groupby(df["Дата"].dt.to_period("M"))["Количество"].apply(
    lambda x: (x == 0).mean()).mul(100).round(1)
print(by_month.to_string())

print("\nпо товарам — 3 самых редких и 3 самых частых нуля:")
by_prod = df.groupby("Номенклатура")["Количество"].apply(
    lambda x: (x == 0).mean()).mul(100).round(1).sort_values()
print(pd.concat([by_prod.head(3), by_prod.tail(3)]).to_string())

Доля нулей монотонно падает с 14.3% в январе до 8.3% в августе и сильно различается
по товарам: от 6.4% у product_1 до 23.3% у product_12.

Это не техническая ошибка. Строка с нулём означает, что позиция была в заявке,
но не отгружена — то есть это фиксация неудовлетворённого спроса. Нули оставляем
в данных: они несут информацию, а их удаление завысило бы средний размер отгрузки.

## 3. Дневная динамика

In [ ]:
grouped_df = df.groupby("Дата", as_index=False)["Количество"].sum()
grouped_df.columns = ["Дата", "Количество продаж"]

print(grouped_df["Количество продаж"].describe().round(1).to_string())
grouped_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))

ax.plot(grouped_df["Дата"], grouped_df["Количество продаж"],
        lw=1, color=GREY, alpha=.75, label="Продажи за день")
roll = grouped_df.set_index("Дата")["Количество продаж"].rolling(28, center=True).mean()
ax.plot(roll.index, roll.values, lw=3, color=NAVY, label="Скользящее среднее, 28 дней")

for _, r in grouped_df.nlargest(2, "Количество продаж").iterrows():
    ax.scatter(r["Дата"], r["Количество продаж"], s=70, color=ORANGE, zorder=5)
    ax.annotate(f"{r['Дата']:%d.%m}\n{int(r['Количество продаж']):,}".replace(",", " "),
                (r["Дата"], r["Количество продаж"]), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=9, color=ORANGE, fontweight="bold")
low = grouped_df.nsmallest(1, "Количество продаж").iloc[0]
ax.scatter([low["Дата"]], [low["Количество продаж"]], s=70, color=ORANGE, zorder=5)
ax.annotate(f"{low['Дата']:%d.%m}\n{int(low['Количество продаж']):,}".replace(",", " "),
            (low["Дата"], low["Количество продаж"]), textcoords="offset points",
            xytext=(0, -32), ha="center", fontsize=9, color=ORANGE, fontweight="bold")

ax.set_title("Дневные продажи: январь — август 2018")
ax.set_ylabel("Продано, шт")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: RU_MON.get(mdates.num2date(x).month, "")))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: f"{int(x):,}".replace(",", " ")))
ax.legend(frameon=False, loc="upper left")
plt.show()

### Что видно на графике

**Ряд растёт, но не плавно.** Скользящее среднее держится около 3 900 с января по
двадцатые числа апреля, затем за две недели поднимается примерно до 4 700 и после
этого растёт медленно. Это не постепенный тренд, а ступенька: уровень сменился
один раз и закрепился.

**Размах дневных колебаний велик** — от 2 326 до 6 226 при среднем 4 339
и стандартном отклонении 648. Пиковый день выше минимального в 2.7 раза.

**Видна регулярная пила** — колебания с периодом около недели.
Ниже проверим, что это дни недели.

**Три дня выбиваются:** 18.04 (2 326 — минимум), 31.07 (6 217) и 21.08
(6 226 — максимум). Оба пика приходятся на конец месяца.

**Тренд к концу периода не выходит на плато** — август остаётся самым сильным месяцем,
значит рост на момент последнего наблюдения не исчерпан.

In [ ]:
# Раскладываем ряд на месячный уровень и недельную сезонность
month_stats = df.groupby(df["Дата"].dt.to_period("M")).agg(
    продано=("Количество", "sum"), дней=("Дата", "nunique"))
month_stats["в день"] = (month_stats["продано"] / month_stats["дней"]).round(0)
month_stats["к январю"] = (month_stats["в день"] / month_stats["в день"].iloc[0] - 1
                           ).mul(100).round(1)
print(month_stats.to_string())

gd = grouped_df.copy()
gd["день недели"] = gd["Дата"].dt.dayofweek
dow = gd.groupby("день недели")["Количество продаж"].agg(["mean", "count"])
dow.index = [DAYS[i] for i in dow.index]
dow["отклонение, %"] = (dow["mean"] / gd["Количество продаж"].mean() - 1).mul(100).round(1)
print("\n" + dow.round(0).to_string())

Вторник даёт на 13% больше среднего, суббота — на 8% меньше. Разрыв между лучшим
и худшим днём недели — 23%. При этом понедельника в данных нет, и вторничный пик,
скорее всего, вбирает в себя отложенный за выходной спрос.

## 4. Выбросы

Задача — найти строку с максимальным выбросом по количеству продаж. Но полезно
сразу развести два разных вопроса: аномальная **строка** и аномальный **день** —
это не одно и то же, и ниже видно, почему.

In [ ]:
# Выброс на уровне строки
outlier_row = df.loc[df["Количество"].idxmax()]
print("Строка с максимальным количеством:")
print(outlier_row.to_string())

q1, q3 = df["Количество"].quantile([.25, .75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f"\nГраница выбросов по правилу Тьюки: Q3 + 1.5·IQR = {upper:.1f}")
n_out = (df["Количество"] > upper).sum()
print(f"строк выше границы: {n_out:,}".replace(",", " ")
      + f" ({n_out / len(df):.1%}), они дают "
      f"{df.loc[df['Количество'] > upper, 'Количество'].sum() / df['Количество'].sum():.1%} объёма")

print("\nВсе строки с количеством 100 и выше:")
print(df[df["Количество"] >= 100].sort_values("Количество", ascending=False).to_string(index=False))

In [ ]:
# Что это за контрагент и был ли аномальным сам день
client = outlier_row["Контрагент"]
hist = df[df["Контрагент"] == client].groupby("Дата")["Количество"].sum()
print(f"{client}: дней с покупками {len(hist)}, "
      f"медиана {hist.median():.0f}, максимум {hist.max()}")

day_total = grouped_df.loc[grouped_df["Дата"] == outlier_row["Дата"],
                           "Количество продаж"].iloc[0]
s = grouped_df["Количество продаж"]
z = (day_total - s.mean()) / s.std()
print(f"\nпродажи {outlier_row['Дата']:%d.%m.%Y} всего: {day_total:,}".replace(",", " "))
print(f"z-оценка этого дня: {z:.2f}")
print(f"без строки-выброса: {day_total - outlier_row['Количество']:,}".replace(",", " "))

Строка на 200 единиц — действительно максимум по массиву, в 68 раз выше медианы (3 шт).
Контрагент address_208 закупался всего 6 дней за 8 месяцев, то есть это не регулярный
клиент, а разовый крупный покупатель.

**Но день 28.06 при этом совершенно обычный:** 4 246 единиц, z-оценка −0.14 —
ниже среднего. Строка-выброс составляет 4.7% дневного объёма и не делает день аномальным.

Отсюда практический вывод: искать выбросы нужно на том уровне агрегации, на котором
принимается решение. Аномальная строка — это вопрос к сделке и к клиенту;
аномальный день — вопрос к загрузке склада.

In [ ]:
# Выбросы на уровне дня — другой ответ
q1d, q3d = s.quantile([.25, .75])
iqrd = q3d - q1d
lo, hi = q1d - 1.5 * iqrd, q3d + 1.5 * iqrd
day_outliers = grouped_df[(s < lo) | (s > hi)].copy()
day_outliers["z"] = ((day_outliers["Количество продаж"] - s.mean()) / s.std()).round(2)
day_outliers["день недели"] = day_outliers["Дата"].dt.dayofweek.map(lambda i: DAYS[i])
print(f"границы: [{lo:.0f}; {hi:.0f}]")
print(day_outliers.to_string(index=False))

## 5. Где сменился уровень продаж

На графике видна ступенька в конце апреля. Найдём её точку формально: переберём все
возможные точки разделения ряда и выберем ту, где различие средних максимально.

In [ ]:
best_date, best_t = None, 0
for i in range(20, len(s) - 20):
    t_stat, _ = stats.ttest_ind(s.iloc[:i], s.iloc[i:], equal_var=False)
    if abs(t_stat) > abs(best_t):
        best_date, best_t = grouped_df["Дата"].iloc[i], t_stat
print(f"точка перелома: {best_date:%d.%m.%Y}, t = {best_t:.2f}")

before = s[grouped_df["Дата"] < "2018-05-01"]
after = s[grouped_df["Дата"] >= "2018-05-01"]
t_stat, p_value = stats.ttest_ind(before, after, equal_var=False)
print(f"\nянварь — апрель: {before.mean():.0f} в день (n = {len(before)})")
print(f"май — август:     {after.mean():.0f} в день (n = {len(after)})")
print(f"разница: {after.mean() - before.mean():+.0f} ({after.mean() / before.mean() - 1:+.1%})")
print(f"t-критерий Уэлча: t = {t_stat:.2f}, p-value = {p_value:.2e}")

In [ ]:
# За счёт чего вырос объём: больше отгрузок или крупнее каждая?
pre = df[(df["Дата"] >= "2018-03-01") & (df["Дата"] < "2018-05-01")]
post = df[(df["Дата"] >= "2018-05-01") & (df["Дата"] < "2018-07-01")]

d_pre, d_post = pre["Дата"].nunique(), post["Дата"].nunique()
rows_pre, rows_post = len(pre) / d_pre, len(post) / d_post
avg_pre, avg_post = pre["Количество"].mean(), post["Количество"].mean()

print(f"{'':22s}{'мар-апр':>10s}{'май-июн':>10s}{'изменение':>12s}")
print(f"{'строк в день':22s}{rows_pre:10.0f}{rows_post:10.0f}{rows_post / rows_pre - 1:>11.1%}")
print(f"{'среднее в строке':22s}{avg_pre:10.2f}{avg_post:10.2f}{avg_post / avg_pre - 1:>11.1%}")
print(f"{'контрагентов':22s}{pre['Контрагент'].nunique():10d}"
      f"{post['Контрагент'].nunique():10d}")
print(f"{'продано в день':22s}{pre['Количество'].sum() / d_pre:10.0f}"
      f"{post['Количество'].sum() / d_post:10.0f}"
      f"{(post['Количество'].sum() / d_post) / (pre['Количество'].sum() / d_pre) - 1:>11.1%}")

Рост на 17.6% почти целиком объясняется размером отгрузки: средняя строка выросла
с 2.74 до 3.15 единиц (+15.2%), тогда как число строк в день прибавило лишь 2.1%,
а число контрагентов — с 192 до 199.

То есть клиентская база не расширилась. Те же клиенты стали брать больше за раз.
Это важное различие: привлечения не было, был рост потребления или укрупнение заказа.

## 6. Топовый товар: склад 3, среды, июнь — август

In [ ]:
mask = (
    (df["Склад"] == 3)
    & (df["Дата"].dt.dayofweek == 2)          # среда
    & (df["Дата"].dt.month.isin([6, 7, 8]))   # июнь, июль, август
)
subset = df[mask]
print(f"строк в срезе: {len(subset):,}".replace(",", " "))
print(f"сред в выборке: {subset['Дата'].nunique()}")
print(f"продано в срезе: {subset['Количество'].sum():,}".replace(",", " "))

top = (subset.groupby("Номенклатура")["Количество"]
       .agg(продано="sum", среднее="mean", строк="size")
       .sort_values("продано", ascending=False))
top["доля, %"] = (top["продано"] / top["продано"].sum() * 100).round(2)
top.head(8).round(2)

In [ ]:
answer = top.index[0]
print(f"Ответ: {answer} — {top['продано'].iloc[0]:,} шт".replace(",", " ")
      + f" ({top['доля, %'].iloc[0]}% среза)")
print(f"второе место: {top.index[1]} — {top['продано'].iloc[1]:,} шт".replace(",", " ")
      + f", отрыв лидера {top['продано'].iloc[0] / top['продано'].iloc[1] - 1:+.1%}")

# Насколько ответ устойчив к изменению среза
print("\nПроверка устойчивости — тот же вопрос на других срезах:")
variants = {
    "все склады, среды, июнь-авг":
        (df["Дата"].dt.dayofweek == 2) & df["Дата"].dt.month.isin([6, 7, 8]),
    "склад 3, все дни, июнь-авг":
        (df["Склад"] == 3) & df["Дата"].dt.month.isin([6, 7, 8]),
    "склад 3, среды, весь период":
        (df["Склад"] == 3) & (df["Дата"].dt.dayofweek == 2),
    "весь массив":
        pd.Series(True, index=df.index),
}
for label, m in variants.items():
    r = df[m].groupby("Номенклатура")["Количество"].sum().sort_values(ascending=False)
    print(f"  {label:32s} → {r.index[0]} ({r.iloc[0]:,})".replace(",", " "))

Ответ — **product_1**, 2 267 штук, 21.6% объёма среза. Отрыв от product_2 составляет
всего 10%, то есть лидерство неуверенное: на 13 наблюдениях (столько сред попало
в июнь — август) такой разрыв мог возникнуть случайно.

Проверка на других срезах показывает, что product_1 лидирует везде — и по всем складам,
и по всем дням недели, и на всём периоде. Значит, узкий срез из задачи не выявляет
никакой локальной специфики: он просто воспроизводит общего лидера продаж.

## 7. Структура продаж: товары и клиенты

In [ ]:
abc = (df.groupby("Номенклатура")["Количество"].sum()
       .sort_values(ascending=False).to_frame("продано"))
abc["доля, %"] = (abc["продано"] / abc["продано"].sum() * 100).round(2)
abc["накопл., %"] = abc["доля, %"].cumsum().round(2)
abc["группа"] = pd.cut(abc["накопл., %"], [0, 80, 95, 100], labels=["A", "B", "C"])

print(abc.groupby("группа", observed=True).agg(
    товаров=("продано", "size"), доля=("доля, %", "sum")).round(1).to_string())
abc.head(10)

In [ ]:
clients = df.groupby("Контрагент")["Количество"].sum().sort_values(ascending=False)
print(f"контрагентов: {len(clients)}")
for n in (10, 20, 50):
    print(f"  топ-{n:<3d} дают {clients.head(n).sum() / clients.sum():.1%} объёма")
print(f"\nмедиана на контрагента: {clients.median():,.0f}".replace(",", " ")
      + f", среднее: {clients.mean():,.0f}".replace(",", " "))
print(f"минимум: {clients.min()}, максимум: {clients.max():,}".replace(",", " "))

# Появлялись ли новые клиенты
first_purchase = df.groupby("Контрагент")["Дата"].min().dt.to_period("M")
print("\nмесяц первой покупки:")
print(first_purchase.value_counts().sort_index().to_string())

Семь товаров из 24 формируют 78.5% объёма — концентрация высокая, но не экстремальная.

А вот клиентская база, наоборот, **распределена ровно**: топ-10 контрагентов из 211
дают лишь 13.5% объёма, топ-50 — 45.1%. Медиана (3 827) близка к среднему (4 215),
то есть выраженного «ядра» крупных клиентов нет.

Из 211 контрагентов 197 присутствуют с января, за весь период добавилось 14 новых.
Это подтверждает вывод раздела 5: роста за счёт привлечения не было.

## 8. Проверка гипотезы: влияет ли температура на продажи

Продажи растут с января по август. Температура в Астане за те же месяцы растёт
с минус двадцати до плюс двадцати с лишним. Любая корреляция между двумя рядами,
которые оба растут во времени, будет высокой — но это ничего не скажет о влиянии
погоды. Оба ряда просто следуют за календарём.

Поэтому считаем две величины:
- **наивную корреляцию** по сырым рядам — она покажет совместный сезонный ход;
- **корреляцию отклонений от месячного среднего** — в ней общий сезонный тренд
  вычтен из обоих рядов, и остаётся вопрос по существу: в тёплые дни *внутри одного
  месяца* продают больше, чем в холодные?

Именно вторая величина отвечает на вопрос о влиянии погоды.

In [ ]:
def load_weather(path="data/weather_astana_2018.csv"):
    """Средняя суточная температура в Астане, градусы Цельсия.

    Данные из открытого архива Open-Meteo (реанализ ERA5),
    координаты 51.14 N, 71.42 E, часовой пояс Asia/Almaty.
    Сырой экспорт Open-Meteo содержит три служебные строки перед заголовком —
    функция распознаёт оба варианта файла.
    """
    w = pd.read_csv(path)
    if w.columns[0] != "Дата":
        w = pd.read_csv(path, skiprows=3)
        w.columns = ["Дата", "T"]
    w["Дата"] = pd.to_datetime(w["Дата"])
    print(f"загружено дней: {len(w)}, "
          f"период {w['Дата'].min():%d.%m.%Y} — {w['Дата'].max():%d.%m.%Y}")
    return w


weather = load_weather()
weather.describe().round(1)

In [ ]:
merged = grouped_df.merge(weather, on="Дата", how="left")

# После соединения проверяем, что ничего не потерялось и не размножилось
assert len(merged) == len(grouped_df), "merge изменил число строк"
print(f"строк до соединения: {len(grouped_df)}, после: {len(merged)}")
print(f"дней без температуры: {merged['T'].isna().sum()}")

merged = merged.dropna(subset=["T"])
merged.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax = axes[0]
ax.plot(merged["Дата"], merged["Количество продаж"], color=GREY, lw=1, alpha=.7)
ax.plot(merged["Дата"],
        merged.set_index("Дата")["Количество продаж"].rolling(28, center=True).mean().values,
        color=NAVY, lw=2.5)
ax.set_ylabel("Продано, шт", color=NAVY)
ax.set_title("Продажи и температура")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: f"{int(x):,}".replace(",", " ")))

ax = axes[1]
ax.plot(merged["Дата"], merged["T"], color=ORANGE, lw=1.5)
ax.axhline(0, color=GREY, lw=1)
ax.set_ylabel("Температура, °C", color=ORANGE)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: RU_MON.get(mdates.num2date(x).month, "")))
plt.tight_layout()
plt.show()

In [ ]:
# Наивная корреляция по сырым рядам
r_raw, p_raw = stats.pearsonr(merged["T"], merged["Количество продаж"])
rho_raw, _ = stats.spearmanr(merged["T"], merged["Количество продаж"])
print(f"Пирсон по сырым рядам:   r = {r_raw:.3f}  (p = {p_raw:.2e})")
print(f"Спирмен по сырым рядам:  rho = {rho_raw:.3f}")
print(f"R² = {r_raw ** 2:.3f} — доля объяснённой дисперсии\n")

# Корреляция отклонений от месячного среднего
m = merged.copy()
m["месяц"] = m["Дата"].dt.to_period("M")
m["продажи_откл"] = m["Количество продаж"] - m.groupby("месяц")["Количество продаж"].transform("mean")
m["T_откл"] = m["T"] - m.groupby("месяц")["T"].transform("mean")

r_adj, p_adj = stats.pearsonr(m["T_откл"], m["продажи_откл"])
print(f"Пирсон по отклонениям от месячного среднего: r = {r_adj:.3f}  (p = {p_adj:.3f})")
print(f"R² = {r_adj ** 2:.3f}")
print(f"\nмесячный уровень объясняет "
      f"{1 - m['продажи_откл'].var() / m['Количество продаж'].var():.1%} "
      f"дисперсии дневных продаж")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(merged["T"], merged["Количество продаж"], s=22, color=NAVY, alpha=.55)
b, a = np.polyfit(merged["T"], merged["Количество продаж"], 1)
xs = np.linspace(merged["T"].min(), merged["T"].max(), 50)
axes[0].plot(xs, a + b * xs, color=ORANGE, lw=2.5)
axes[0].set_title(f"Сырые ряды: r = {r_raw:.2f}")
axes[0].set_xlabel("Средняя температура за день, °C")
axes[0].set_ylabel("Продано, шт")

axes[1].scatter(m["T_откл"], m["продажи_откл"], s=22, color=GREEN, alpha=.55)
b2, a2 = np.polyfit(m["T_откл"], m["продажи_откл"], 1)
xs2 = np.linspace(m["T_откл"].min(), m["T_откл"].max(), 50)
axes[1].plot(xs2, a2 + b2 * xs2, color=ORANGE, lw=2.5)
axes[1].axhline(0, color=GREY, lw=1)
axes[1].axvline(0, color=GREY, lw=1)
axes[1].set_title(f"Отклонения от месячного среднего: r = {r_adj:.2f}")
axes[1].set_xlabel("Отклонение температуры, °C")
axes[1].set_ylabel("Отклонение продаж, шт")

for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, p: f"{int(x):,}".replace(",", " ")))
plt.tight_layout()
plt.show()

In [ ]:
# Контроль: та же проверка на «плацебо-факторе» — порядковом номере дня.
# Он заведомо не влияет на продажи, но растёт во времени так же, как температура.
merged["номер_дня"] = np.arange(len(merged))
r_placebo, _ = stats.pearsonr(merged["номер_дня"], merged["Количество продаж"])
print(f"корреляция продаж с порядковым номером дня: r = {r_placebo:.3f}")
print("Если корреляция с температурой близка к этой величине, температура")
print("работает не как причина, а как заменитель календаря.")

### Результат: гипотеза не подтвердилась

**Сырые ряды дают r = 0.60 (p ≈ 9·10⁻²²).** Формально — сильная значимая связь.
Именно эту цифру назвали бы ответом, если остановиться здесь.

**Отклонения от месячного среднего дают r = −0.06 (p = 0.38).** Связь исчезает
полностью, и знак меняется на противоположный. Внутри одного месяца тёплые дни
ничем не отличаются от холодных.

**Плацебо-проверка объясняет, почему так.** Порядковый номер дня коррелирует
с продажами на r = 0.69 — *сильнее*, чем температура. Номер дня физически не может
влиять на спрос. Значит, обе корреляции измеряют одно и то же: и продажи,
и температура растут по календарю с января по август.

Температура сама на 90.5% объясняется месяцем — то есть она почти целиком
является календарём, а не самостоятельным фактором.

**Вывод: погода на продажи не влияет.** Рост продаж на 35% объясняется тем,
что произошло 24 апреля, а не потеплением.

In [ ]:
# Дополнительный контроль: частная корреляция при фиксированном номере дня
def partial_corr(x, y, z):
    r_xy = stats.pearsonr(x, y)[0]
    r_xz = stats.pearsonr(x, z)[0]
    r_yz = stats.pearsonr(y, z)[0]
    return (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz ** 2) * (1 - r_yz ** 2))


pr = partial_corr(merged["T"], merged["Количество продаж"], merged["номер_дня"])
n = len(merged)
t_pr = pr * np.sqrt((n - 3) / (1 - pr ** 2))
p_pr = 2 * (1 - stats.t.cdf(abs(t_pr), n - 3))
print(f"частная корреляция T и продаж при фиксированном номере дня: "
      f"r = {pr:.4f} (p = {p_pr:.3f})")

# И с вычетом ещё и дня недели
m["дн"] = m["Дата"].dt.dayofweek
m["продажи_откл2"] = m["продажи_откл"] - m.groupby("дн")["продажи_откл"].transform("mean")
r_dow, p_dow = stats.pearsonr(m["T_откл"], m["продажи_откл2"])
print(f"после вычета месяца и дня недели: r = {r_dow:.4f} (p = {p_dow:.3f})")

print(f"\nдоля дисперсии температуры, объяснённая месяцем: "
      f"{1 - m['T_откл'].var() / m['T'].var():.1%}")

Оба контроля подтверждают вывод: r = −0.08 при фиксированном номере дня (p = 0.23)
и r = −0.04 после вычета месяца и дня недели (p = 0.57). Ни одна оценка
не отличается от нуля значимо.

## Выводы

**Рост на 35% — это одна ступенька в конце апреля, а не плавный тренд.**
Точка перелома — 24.04.2018. Среднедневные продажи: 3 878 в январе — апреле против
4 769 в мае — августе, разница 23% (t-критерий Уэлча, p ≈ 2·10⁻³⁰).

**Вырос размер отгрузки, а не число клиентов.** Средняя строка увеличилась
с 2.74 до 3.15 единиц (+15.2%), число строк в день — всего на 2.1%.
Из 211 контрагентов 197 присутствуют с января, за 8 месяцев добавилось 14.
Привлечения не было — те же клиенты стали брать больше.

**Погода на продажи не влияет.** Сырая корреляция r = 0.60 выглядит убедительно,
но после вычета месячного уровня остаётся r = −0.06 (p = 0.38). Порядковый номер дня
коррелирует с продажами сильнее температуры (r = 0.69), хотя причиной быть не может.
Обе величины измеряют календарь, а не спрос.

**Клиентская база необычно ровная.** Топ-10 контрагентов дают 13.5% объёма,
медиана близка к среднему. Зависимости от нескольких крупных клиентов нет —
это устойчиво, но означает, что и точек роста через ключевых клиентов немного.

**Понедельников в данных нет вообще.** Все 34 понедельника периода отсутствуют.
Это меняет любой расчёт недельных средних и объясняет вторничный пик (+13% к среднему):
вторник вбирает отложенный спрос.

**Аномальная строка и аномальный день — разные вещи.** Максимум по строке (200 шт,
28.06) приходится на день с z-оценкой −0.14, то есть на совершенно рядовой день.
Уровень агрегации нужно выбирать под то решение, которое принимается по результату.

**Неудовлетворённый спрос сокращается.** Доля строк с нулевой отгрузкой упала
с 14.3% в январе до 8.3% в августе. Часть роста продаж может объясняться улучшением
наличия товара, а не увеличением спроса — это стоит проверить отдельно.